# Veritas Phase 1: CIFAKE baseline
Train EfficientNet-B0 on the CIFAKE train split only, evaluate on the official test split, export to ONNX.
Settings: GPU on, Add Input = CIFAKE dataset.

In [ ]:
!git clone https://github.com/YOUR_USERNAME/veritas-detection-engine.git
%cd veritas-detection-engine
!pip install -q onnx onnxruntime

In [ ]:
import glob, os
ROOT = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train', recursive=True)
            if os.path.isdir(os.path.join(os.path.dirname(p), 'test')))
print(ROOT)
!python scripts/make_manifest.py --root {ROOT} --out data/cifake_manifest.csv

In [ ]:
import torch, json
from torch.utils.data import DataLoader
from src.data.manifest import read_manifest, split_rows
from src.data.torch_dataset import ManifestDataset, build_transforms
from src.vision.cnn_2d_artifact import build_model

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
train_rows, val_rows, test_rows = split_rows(read_manifest('data/cifake_manifest.csv'))
print(len(train_rows), len(val_rows), len(test_rows))

def loader(rows, train):
    ds = ManifestDataset(rows, ROOT, build_transforms(train))
    return DataLoader(ds, batch_size=128, shuffle=train, num_workers=2, pin_memory=True)

train_dl, val_dl, test_dl = loader(train_rows, True), loader(val_rows, False), loader(test_rows, False)

In [ ]:
from src.training.train import train
model = build_model()
history = train(model, train_dl, val_dl, device, epochs=3, ckpt_path='/kaggle/working/baseline_best.pt')

## Evaluate on the official CIFAKE test split (run once, after training is finished)

In [ ]:
from src.training.train import predict
from src.eval.metrics import binary_metrics, roc_auc, expected_calibration_error, calibration_bins
import matplotlib.pyplot as plt

model.load_state_dict(torch.load('/kaggle/working/baseline_best.pt'))
probs, labels = predict(model, test_dl, device)
result = {**binary_metrics(labels, probs), 'auc': roc_auc(labels, probs), 'ece': expected_calibration_error(labels, probs)}
print(json.dumps(result, indent=2))
os.makedirs('results', exist_ok=True)
json.dump(result, open('results/cifake_baseline.json', 'w'), indent=2)

b = calibration_bins(labels, probs)
plt.plot([0, 1], [0, 1], '--'); plt.plot([x[0] for x in b], [x[1] for x in b], 'o-')
plt.xlabel('mean predicted P(AI)'); plt.ylabel('observed fraction AI'); plt.title('Calibration (CIFAKE test)'); plt.show()

In [ ]:
from src.training.export import export_onnx
export_onnx(model, '/kaggle/working/image_artifact.onnx')
print(os.path.getsize('/kaggle/working/image_artifact.onnx') / 1e6, 'MB')

Download `baseline_best.pt`, `image_artifact.onnx` and `results/cifake_baseline.json` from the Output panel. Do not commit the weights to git.